<a href="https://colab.research.google.com/github/pyypyyy/aa2codon/blob/main/Codon_optimizer_train_tf219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Codon optimizer training (TensorFlow 2.19+)
This notebook trains a codon optimizer Transformer using TensorFlow 2.19+.
It expects CDS FASTA files in `Genomes/`. Update paths as needed.


In [ ]:
# Optional: install deps in Colab or a clean environment
!pip install "tensorflow==2.19.*" "tensorflow-text==2.19.*" biopython


In [ ]:
!git clone https://github.com/pyypyyy/aa2codon.git

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from Bio import SeqIO
from Bio.Seq import Seq
from tqdm.notebook import tqdm


## Data preparation


In [ ]:
def whitespace_separate(my_string, n):
    return " ".join(my_string[i:i+n] for i in range(0, len(my_string), n))

def process_cds(directory, max_seqs=7000):
    """
    Parse CDS FASTA files, filter pseudogenes and sequences not divisible by 3.
    Returns a dataframe with codon and amino-acid sequences.
    """
    files = [f for f in os.listdir(directory) if not f.startswith('.')]
    cds_proteins = []
    for file in tqdm(files):
        fasta_file = os.path.join(directory, file)
        if not os.path.isfile(fasta_file):
            continue
        records = list(SeqIO.parse(fasta_file, "fasta"))
        skipped = 0
        accepted = 0
        for rec in tqdm(records, total=len(records)):
            if len(rec.seq) % 3 != 0:
                skipped += 1
                continue
            if "pseudogene" in rec.description:
                skipped += 1
                continue
            if accepted >= max_seqs:
                continue
            protein = rec.seq.translate()
            cds_proteins.append((file.split(".")[0], str(rec.seq), str(protein)))
            accepted += 1
        print(f"{file}: skipped {skipped}, accepted {accepted}")

    df = pd.DataFrame(cds_proteins, columns=["filename", "gene_sequence", "protein_sequence"])
    df["gene_sequence"] = df["gene_sequence"].apply(lambda x: whitespace_separate(x, 3))
    df["protein_sequence"] = df["protein_sequence"].apply(lambda x: whitespace_separate(x, 1))

    df["gene_sequence"] = df["gene_sequence"].str.split()
    df["protein_sequence"] = df["protein_sequence"].str.split()

    if len(files) > 1:
        for i in range(len(df)):
            df.at[i, "gene_sequence"].insert(0, f"<{df.at[i, 'filename']}>")
            df.at[i, "protein_sequence"].insert(0, f"<{df.at[i, 'filename']}>")

    df["gene_sequence"] = df["gene_sequence"].str.join(' ')
    df["protein_sequence"] = df["protein_sequence"].str.join(' ')
    return df.drop(columns=["filename"])


In [ ]:
def shorten(seq, max_length):
    seq2 = []
    for i in tqdm(seq):
        if len(i) > max_length:
            organism = i[0]
            first = i[: len(i) // 2]
            second = i[len(i) // 2 :]
            second.insert(0, organism)
            seq2.extend([first, second])
        else:
            seq2.append(i)
    return seq2


In [ ]:
data_path = Path('/content/aa2codon/Genomes')
df = process_cds(str(data_path), max_seqs=7000)
print(df.head())


In [ ]:
max_length = 150
codons = [s.split(' ') for s in df['gene_sequence'].tolist()]
aas = [s.split(' ') for s in df['protein_sequence'].tolist()]

while any(len(x) > max_length for x in codons):
    codons = shorten(codons, max_length)
while any(len(x) > max_length for x in aas):
    aas = shorten(aas, max_length)

for i in codons:
    i.insert(0, '<start>')
    i.append('<end>')
for i in aas:
    i.insert(0, '<start>')
    i.append('<end>')

codons = [' '.join(i) for i in codons]
aas = [' '.join(i) for i in aas]


## Vectorization & datasets


In [ ]:
def makeds(seqs):
    num_val_samples = int(0.07 * len(seqs))
    num_train_samples = len(seqs) - 2 * num_val_samples
    train_pairs = seqs[:num_train_samples]
    val_pairs = seqs[num_train_samples:]
    return train_pairs, val_pairs

train_aas, val_aas = makeds(aas)
train_codons, val_codons = makeds(codons)

tds = tf.data.Dataset.from_tensor_slices((train_aas, train_codons))
vds = tf.data.Dataset.from_tensor_slices((val_aas, val_codons))

aa2id = tf.keras.layers.TextVectorization(standardize=None, split='whitespace')
aa2id.adapt(tf.constant(aas))
codon2id = tf.keras.layers.TextVectorization(standardize=None, split='whitespace')
codon2id.adapt(tf.constant(codons))


In [ ]:
def prepare_batch(aas, codons):
    aas = aa2id(aas)
    codons = codon2id(codons)
    codons_inputs = codons[:, :-1]
    codons_labels = codons[:, 1:]
    return (aas, codons_inputs), codons_labels

def make_batches(ds, buffer_size=100000, batch_size=64):
    return (
        ds.shuffle(buffer_size)
        .batch(batch_size)
        .map(prepare_batch, tf.data.AUTOTUNE)
        .prefetch(buffer_size=tf.data.AUTOTUNE)
    )


## Model


In [ ]:
def positional_encoding(length, depth):
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth
    angle_rates = 1 / (10000 ** depths)
    angle_rads = positions * angle_rates
    pos_encoding = np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    return tf.cast(pos_encoding, dtype=tf.float32)

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True)
        self.pos_encoding = positional_encoding(length=2048, depth=d_model)

    def compute_mask(self, *args, **kwargs):
        return self.embedding.compute_mask(*args, **kwargs)

    def call(self, x):
        length = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[tf.newaxis, :length, :]
        return x


In [ ]:
class BaseAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
        self.layernorm = tf.keras.layers.LayerNormalization()
        self.add = tf.keras.layers.Add()

class CrossAttention(BaseAttention):
    def call(self, x, context):
        attn_output, attn_scores = self.mha(
            query=x, key=context, value=context, return_attention_scores=True
        )
        self.last_attn_scores = attn_scores
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

class GlobalSelfAttention(BaseAttention):
    def call(self, x):
        attn_output = self.mha(query=x, value=x, key=x)
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

class CausalSelfAttention(BaseAttention):
    def call(self, x):
        attn_output = self.mha(query=x, value=x, key=x, use_causal_mask=True)
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

class FeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, dff, dropout_rate=0.1):
        super().__init__()
        self.seq = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model),
            tf.keras.layers.Dropout(dropout_rate),
        ])
        self.add = tf.keras.layers.Add()
        self.layer_norm = tf.keras.layers.LayerNormalization()

    def call(self, x):
        x = self.add([x, self.seq(x)])
        x = self.layer_norm(x)
        return x


In [ ]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, *, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.self_attention = GlobalSelfAttention(
            num_heads=num_heads, key_dim=d_model, dropout=dropout_rate
        )
        self.ffn = FeedForward(d_model, dff)

    def call(self, x):
        x = self.self_attention(x)
        x = self.ffn(x)
        return x

class Encoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size, d_model=d_model)
        self.enc_layers = [
            EncoderLayer(d_model=d_model, num_heads=num_heads, dff=dff, dropout_rate=dropout_rate)
            for _ in range(num_layers)
        ]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x):
        x = self.pos_embedding(x)
        x = self.dropout(x)
        for i in range(self.num_layers):
            x = self.enc_layers[i](x)
        return x

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, *, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.causal_self_attention = CausalSelfAttention(
            num_heads=num_heads, key_dim=d_model, dropout=dropout_rate
        )
        self.cross_attention = CrossAttention(
            num_heads=num_heads, key_dim=d_model, dropout=dropout_rate
        )
        self.ffn = FeedForward(d_model, dff)

    def call(self, x, context):
        x = self.causal_self_attention(x=x)
        x = self.cross_attention(x=x, context=context)
        self.last_attn_scores = self.cross_attention.last_attn_scores
        x = self.ffn(x)
        return x

class Decoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size, d_model=d_model)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.dec_layers = [
            DecoderLayer(d_model=d_model, num_heads=num_heads, dff=dff, dropout_rate=dropout_rate)
            for _ in range(num_layers)
        ]
        self.last_attn_scores = None

    def call(self, x, context):
        x = self.pos_embedding(x)
        x = self.dropout(x)
        for i in range(self.num_layers):
            x = self.dec_layers[i](x, context)
        self.last_attn_scores = self.dec_layers[-1].last_attn_scores
        return x

class Transformer(tf.keras.Model):
    def __init__(self, *, num_layers, d_model, num_heads, dff, input_vocab_size, target_vocab_size, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(
            num_layers=num_layers, d_model=d_model, num_heads=num_heads, dff=dff,
            vocab_size=input_vocab_size, dropout_rate=dropout_rate
        )
        self.decoder = Decoder(
            num_layers=num_layers, d_model=d_model, num_heads=num_heads, dff=dff,
            vocab_size=target_vocab_size, dropout_rate=dropout_rate
        )
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs):
        context, x = inputs
        context = self.encoder(context)
        x = self.decoder(x, context)
        logits = self.final_layer(x)
        try:
            del logits._keras_mask
        except AttributeError:
            pass
        return logits


## Training


In [ ]:
codon_list = [
    'CTG','GAA','GCG','ATG','AAA','GAT','ATT','GGC','CAG','GTG','GCC',
    'ATC','GGT','ACC','CCG','TTT','CGC','AAC','CGT','GCA','GAC','GTT','GAG','AAT','TTC',
    'AGC','TAT','CAA','GTC','GCT','TGG','ACG','TTG','TTA','CAT','TAC','CTC','CTT','GGG',
    'GTA','AAG','CAC','TCG','ACT','AGT','TCC','TCT','CCA','GGA','TCA',
    'CCT','ACA','TGC','CCC','CGG','TGT','ATA','CTA','CGA','TAA','AGA','AGG','TGA','TAG'
]

def make_table(keys, values):
    init = tf.lookup.KeyValueTensorInitializer(keys, values)
    return tf.lookup.StaticHashTable(init, default_value=-1)

aa_table = [str(Seq(codon).translate()) for codon in codon_list]
codon_table = codon_list + ['<start>', '<end>', '[UNK]']
aa_table = aa_table + ['<start>', '<end>', '[UNK]']

codon_table = [codon2id(codon) for codon in codon_table]
aa_table = [aa2id(aa) for aa in aa_table]

codon_table.append(tf.convert_to_tensor([0], dtype=tf.int64))
aa_table.append(tf.convert_to_tensor([0], dtype=tf.int64))

tensor_codon_to_tensor_aa = make_table(codon_table, aa_table)


In [ ]:
def masked_loss(label, pred):
    mask = label != 0
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    loss = loss_object(label, pred)
    mask = tf.cast(mask, dtype=loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(label, pred):
    pred = tf.argmax(pred, axis=2)
    label = tf.cast(label, pred.dtype)
    match = label == pred
    mask = label != 0
    match = match & mask
    match = tf.cast(match, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(match) / tf.reduce_sum(mask)

def masked_aa_accuracy(label, pred):
    pred = tf.argmax(pred, axis=2)
    pred = tensor_codon_to_tensor_aa.lookup(pred)
    label = tf.cast(label, pred.dtype)
    label = tensor_codon_to_tensor_aa.lookup(label)
    match = label == pred
    mask = label != 0
    match = match & mask
    match = tf.cast(match, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(match) / tf.reduce_sum(mask)


In [ ]:
num_layers = 3
d_model = 128
dff = 256
num_heads = 8
dropout_rate = 0.05

transformer = Transformer(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    input_vocab_size=len(aa2id.get_vocabulary()),
    target_vocab_size=len(codon2id.get_vocabulary()),
    dropout_rate=dropout_rate,
)


In [ ]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, dtype=tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

learning_rate = CustomSchedule(d_model)
optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate, beta_1=0.9, beta_2=0.9, epsilon=1e-9
)

transformer.compile(
    loss=masked_loss,
    optimizer=optimizer,
    metrics=[masked_accuracy, masked_aa_accuracy],
)


In [ ]:
train_batches = make_batches(tds)
val_batches = make_batches(vds)

callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True, verbose=1
)

history = transformer.fit(
    train_batches,
    epochs=100,
    validation_data=val_batches,
    callbacks=[callback],
)


## Export


In [ ]:
class Translator(tf.Module):
    def __init__(self, aa2id, codon2id, transformer):
        self.aa2id = aa2id
        self.codon2id = codon2id
        self.transformer = transformer

    def __call__(self, sentence, max_length=max_length):
        sentence = self.aa2id(sentence)[tf.newaxis]
        encoder_input = sentence
        start_end = self.codon2id('<start> <end>')
        start = start_end[0][tf.newaxis]
        end = start_end[1][tf.newaxis]
        output_array = tf.TensorArray(dtype=tf.int64, size=0, dynamic_size=True)
        output_array = output_array.write(0, start)
        for i in tf.range(max_length):
            output = tf.transpose(output_array.stack())
            predictions = self.transformer([encoder_input, output], training=False)
            predictions = predictions[:, -1:, :]
            predicted_id = tf.argmax(predictions, axis=-1)
            output_array = output_array.write(i + 1, predicted_id[0])
            if tf.reduce_any(tf.equal(predicted_id, end)):
                break
        output = tf.transpose(output_array.stack())
        self.transformer([encoder_input, output[:, :-1]], training=False)
        attention_weights = self.transformer.decoder.last_attn_scores
        return output[0], attention_weights

class ExportTranslator(tf.Module):
    def __init__(self, translator):
        self.translator = translator

    @tf.function(input_signature=[tf.TensorSpec(shape=[], dtype=tf.string)])
    def __call__(self, sentence):
        result, attention_weights = self.translator(sentence, max_length=max_length)
        return result, attention_weights

translator = Translator(aa2id, codon2id, transformer)
exporter = ExportTranslator(translator)
save_path = 'transformer_tf219'
tf.saved_model.save(exporter, save_path)
print(f'Saved model to {save_path}')
